# 🧬 Breast Cancer Classification — Machine Learning Project

**Objective.** Build a reproducible pipeline to distinguish **malignant** from **benign** tumors using numerical measurements, compare several models, select a model without data leakage, and analyze its errors.

> **Important limitation:** This is an educational project using the public *Breast Cancer Wisconsin* dataset included in scikit-learn. The model is neither a medical device nor a real diagnostic tool. Performance on this dataset does not guarantee generalization to other hospitals or populations.

1. Understand the problem and audit the data
2. Exploratory data analysis (quality, distributions, correlations)
3. Split training and test data before any fitting
4. Integrate preprocessing into pipelines
5. Compare models using stratified cross-validation
6. Tune hyperparameters using the training set only
7. Perform a single final evaluation on the test set
8. Interpretability, limitations, and deployment considerations

## 0. Configuration and Reproducibility

Plots use a consistent layout. The random seed is fixed to make results easier to reproduce.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay, RocCurveDisplay,
    PrecisionRecallDisplay, make_scorer
)
from sklearn.inspection import permutation_importance
from sklearn.calibration import CalibrationDisplay
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score

SEED = 42
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 40)
pd.set_option("display.precision", 3)

print("Environment ready.")

## 1. Loading and Understanding the Data

The dataset contains 569 observations and 30 numerical features. The scikit-learn target is encoded as `0 = malignant` and `1 = benign`. For metrics focused on cancer detection, we explicitly define the positive class as **malignant** to avoid ambiguity.

In [ ]:
dataset = load_breast_cancer(as_frame=True)
X = dataset.data.copy()
y_original = dataset.target.copy()

# Explicit target: 1 = malignant, 0 = benign
y = (y_original == 0).astype(int).rename("malignant")

print(f"Observations: {X.shape[0]:,}")
print(f"Features: {X.shape[1]}")
print(f"Memory usage: {X.memory_usage(deep=True).sum() / 1024:.1f} KB")
display(pd.DataFrame({
    "Item": ["Observations", "Features", "Missing values", "Duplicate rows"],
    "Value": [len(X), X.shape[1], int(X.isna().sum().sum()), int(X.duplicated().sum())]
}))

In [ ]:
# Feature dictionary and preview
display(X.head())
display(X.describe().T.rename_axis("Feature"))

### 1.1 Target Distribution

Class imbalance can make accuracy misleading. We examine the absolute and relative frequency of each class before selecting evaluation metrics.

In [ ]:
target_counts = y.map({1: "Malignant", 0: "Benign"}).value_counts().reindex(["Benign", "Malignant"])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
target_counts.plot(kind="bar", ax=axes[0], rot=0)
axes[0].set_title("Number of observations per class")
axes[0].set_ylabel("Count")
for p in axes[0].patches:
    axes[0].annotate(f"{int(p.get_height())}", (p.get_x()+p.get_width()/2, p.get_height()),
                     ha="center", va="bottom", xytext=(0, 3), textcoords="offset points")
axes[1].pie(target_counts.values, labels=target_counts.index, autopct="%1.1f%%", startangle=90)
axes[1].set_title("Class proportions")
plt.tight_layout()
plt.show()

## 2. Data Quality Audit and Exploratory Analysis

The checks cover missing values, duplicates, constant features, and distributions. Outliers are not removed automatically: in a biomedical context, they may contain meaningful signal.

In [ ]:
missing = X.isna().sum().sort_values(ascending=False)
quality = pd.DataFrame({
    "Data type": X.dtypes.astype(str),
    "Missing values": X.isna().sum(),
    "% missing": (X.isna().mean()*100),
    "Unique values": X.nunique(),
    "Standard deviation": X.std()
})
display(quality.sort_values(["Missing values", "Unique values"], ascending=[False, True]).head(12))
print("Number of constant columns:", int((X.nunique() <= 1).sum()))
print("Number of fully duplicated rows:", int(X.duplicated().sum()))

In [ ]:
# Feature distributions: scales and skewness
X.hist(figsize=(18, 15), bins=25, edgecolor="white")
plt.suptitle("Distributions of the 30 features", y=1.01, fontsize=16)
plt.tight_layout()
plt.show()

### 2.1 Feature Comparisons by Diagnosis

We examine several representative features. These plots help generate hypotheses; they do not establish causal relationships.

In [ ]:
explore_cols = ["mean radius", "mean texture", "mean perimeter", "mean area",
                "mean smoothness", "mean concavity"]
plot_df = X[explore_cols].copy()
plot_df["Diagnosis"] = y.map({1: "Malignant", 0: "Benign"})
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col in zip(axes.flat, explore_cols):
    sns.violinplot(data=plot_df, x="Diagnosis", y=col, inner="quartile", cut=0, ax=ax)
    ax.set_title(col.replace("_", " ").title())
plt.suptitle("Feature distributions by diagnosis", y=1.02, fontsize=16)
plt.tight_layout()
plt.show()

### 2.2 Correlations

Highly correlated features may provide redundant information. Correlation does not imply causation; it is mainly useful for understanding the data and the limitations of directly interpreting model coefficients.

In [ ]:
corr = X.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(15, 12))
sns.heatmap(corr, cmap="vlag", center=0, ax=ax, xticklabels=True, yticklabels=True,
            cbar_kws={"shrink": .75})
ax.set_title("Feature correlation matrix")
plt.xticks(rotation=90, fontsize=7)
plt.yticks(fontsize=7)
plt.tight_layout()
plt.show()

corr_target = X.corrwith(y).sort_values()
display(pd.DataFrame({"Correlation with the malignant class": corr_target}).assign(
    Absolute_value=lambda d: d.iloc[:, 0].abs()
).sort_values("Absolute_value", ascending=False).drop(columns="Absolute_value").head(12))

## 3. Data Splitting — Preventing Data Leakage

The test set is held out now. All learned transformations (imputation and standardization) are placed inside pipelines and refitted within each cross-validation fold. The final test set will not be used for feature selection or hyperparameter tuning.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print("\nClass proportions (malignant class)")
display(pd.DataFrame({
    "Train": y_train.value_counts(normalize=True).sort_index(),
    "Test": y_test.value_counts(normalize=True).sort_index()
}, index=[0, 1]).rename(index={0: "Benign (0)", 1: "Malignant (1)"}))

## 4. Modeling and Evaluation Protocol

We compare a baseline with several model families. Models sensitive to feature scales use standardization. Median imputation is included for robustness, even though the audit indicates that there are no missing values.

**Priority metrics:**
- **Malignant recall:** proportion of malignant cases correctly detected.
- **Malignant precision:** among predictions labeled malignant, the proportion that are truly malignant.
- **Malignant F1-score:** balance between precision and recall.
- **ROC-AUC** and **Average Precision:** quality of the ranking of cases.
- **Balanced accuracy:** average recall across the two classes.

No single metric is sufficient for clinical use.

In [ ]:
def pipeline_standardize(model):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", model)
    ])

models = {
    "Majority-class baseline": Pipeline([("imputer", SimpleImputer(strategy="median")),
                                      ("model", DummyClassifier(strategy="most_frequent"))]),
    "Logistic regression": pipeline_standardize(LogisticRegression(max_iter=5000, class_weight="balanced", random_state=SEED)),
    "SVM RBF": pipeline_standardize(SVC(C=1.0, kernel="rbf", probability=True, class_weight="balanced", random_state=SEED)),
    "K-nearest neighbors": pipeline_standardize(KNeighborsClassifier(n_neighbors=7)),
    "Gaussian Naive Bayes": pipeline_standardize(GaussianNB()),
    "Decision tree": Pipeline([("imputer", SimpleImputer(strategy="median")),
                                  ("model", DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=SEED))]),
    "Random forest": Pipeline([("imputer", SimpleImputer(strategy="median")),
                                 ("model", RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                                                  min_samples_leaf=2, random_state=SEED, n_jobs=-1))]),
    "Extra Trees": Pipeline([("imputer", SimpleImputer(strategy="median")),
                             ("model", ExtraTreesClassifier(n_estimators=300, class_weight="balanced",
                                                            min_samples_leaf=2, random_state=SEED, n_jobs=-1))]),
    "Gradient Boosting": Pipeline([("imputer", SimpleImputer(strategy="median")),
                                   ("model", GradientBoostingClassifier(random_state=SEED))]),
    "Hist Gradient Boosting": Pipeline([("imputer", SimpleImputer(strategy="median")),
                                        ("model", HistGradientBoostingClassifier(random_state=SEED))]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
    "malignant_recall": make_scorer(recall_score, pos_label=1, zero_division=0),
    "malignant_precision": make_scorer(precision_score, pos_label=1, zero_division=0),
    "malignant_f1": make_scorer(f1_score, pos_label=1, zero_division=0),
}

In [ ]:
rows = []
for name, estimator in models.items():
    scores = cross_validate(estimator, X_train, y_train, cv=cv, scoring=scoring,
                            n_jobs=-1, return_train_score=False)
    row = {"Model": name}
    for metric in scoring:
        values = scores[f"test_{metric}"]
        row[f"{metric} mean"] = values.mean()
        row[f"{metric} standard deviation"] = values.std()
    rows.append(row)

comparison = pd.DataFrame(rows).sort_values("roc_auc mean", ascending=False).set_index("Model")
display(comparison.round(3))

In [ ]:
# Plot comparison of three important metrics
metrics_plot = ["roc_auc mean", "malignant_recall mean", "malignant_f1 mean"]
plot_compare = comparison[metrics_plot].sort_values("roc_auc mean")
ax = plot_compare.plot(kind="barh", figsize=(12, 7))
ax.set_xlim(0, 1.05)
ax.set_xlabel("Mean cross-validation score")
ax.set_ylabel("")
ax.set_title("Model comparison (training set only)")
ax.legend(["ROC-AUC", "Malignant recall", "Malignant F1-score"], loc="lower right")
plt.tight_layout()
plt.show()

## 5. Model Tuning

We tune logistic regression using a hyperparameter grid and stratified cross-validation. The selection criterion is malignant-class recall, while also monitoring AUC. This choice illustrates a priority of reducing false negatives; it is not a universal clinical recommendation.

In [ ]:
pipeline_lr = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=8000, class_weight="balanced", random_state=SEED))
])
param_grid = {
    "model__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "model__solver": ["liblinear", "lbfgs"],
}
search = GridSearchCV(
    pipeline_lr, param_grid, scoring="recall", cv=cv, n_jobs=-1,
    refit=True, return_train_score=True
)
search.fit(X_train, y_train)
print("Best parameters:", search.best_params_)
print(f"Best cross-validation recall: {search.best_score_:.3f}")

In [ ]:
grid_results = pd.DataFrame(search.cv_results_)
grid_results["C"] = grid_results["param_model__C"].astype(float)
fig, ax = plt.subplots(figsize=(9, 5))
for solver, group in grid_results.groupby("param_model__solver"):
    means = group.groupby("C")["mean_test_score"].mean().sort_index()
    ax.plot(means.index, means.values, marker="o", label=solver)
ax.set_xscale("log")
ax.set_ylim(0, 1.02)
ax.set_xlabel("C (logarithmic scale)")
ax.set_ylabel("Mean malignant recall")
ax.set_title("Hyperparameter search — cross-validation")
ax.legend(title="Solver")
plt.tight_layout()
plt.show()

## 6. Final Evaluation on the Test Set

This section evaluates the selected model on the held-out data. The target is defined as `1 = malignant`, so the probability used corresponds to class `1`. Results should be interpreted cautiously given the modest dataset size.

In [ ]:
best_model = search.best_estimator_
y_pred = best_model.predict(X_test)
y_proba_malignant = best_model.predict_proba(X_test)[:, list(best_model.classes_).index(1)]

final_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Balanced accuracy": balanced_accuracy_score(y_test, y_pred),
    "Malignant precision": precision_score(y_test, y_pred, zero_division=0),
    "Malignant recall": recall_score(y_test, y_pred, zero_division=0),
    "Malignant F1-score": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC (malignant positive)": roc_auc_score(y_test, y_proba_malignant),
    "Average Precision": average_precision_score(y_test, y_proba_malignant),
}
display(pd.DataFrame.from_dict(final_metrics, orient="index", columns=["Score"]))
print("\nClassification report (1 = malignant):")
print(classification_report(y_test, y_pred, target_names=["Benign", "Malignant"], zero_division=0))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
ConfusionMatrixDisplay(cm, display_labels=["Benign", "Malignant"]).plot(ax=axes[0], colorbar=False)
axes[0].set_title("Confusion matrix")

RocCurveDisplay.from_predictions(y_test, y_proba_malignant, ax=axes[1], name="Final model")
axes[1].plot([0, 1], [0, 1], linestyle="--", linewidth=1)
axes[1].set_title("ROC curve")

PrecisionRecallDisplay.from_predictions(y_test, y_proba_malignant, ax=axes[2], name="Final model")
axes[2].set_title("Precision-recall curve")
plt.tight_layout()
plt.show()

### 6.1 Error Analysis

False negatives are malignant cases predicted as benign; false positives are benign cases predicted as malignant. These two types of errors have different consequences. The model must not be used to make medical decisions.

In [ ]:
errors = X_test.copy()
errors["Actual"] = y_test.map({0: "Benign", 1: "Malignant"})
errors["Predicted"] = pd.Series(y_pred, index=y_test.index).map({0: "Benign", 1: "Malignant"})
errors["Malignant probability"] = y_proba_malignant
errors["Error type"] = np.select(
    [(y_test.values == 1) & (y_pred == 0), (y_test.values == 0) & (y_pred == 1)],
    ["False negative", "False positive"], default="Correct"
)
display(errors.loc[errors["Error type"] != "Correct",
                   ["Actual", "Predicted", "Malignant probability", "Error type"]].sort_values("Malignant probability"))

## 7. Interpretability: Permutation Importance

Permutation importance measures the decrease in performance when a feature is shuffled. It is calculated on the test set here as a post-evaluation analysis; it must not be used to modify the model and then report the same test scores again. Correlated features may share or mask one another's importance.

In [ ]:
perm = permutation_importance(
    best_model, X_test, y_test, scoring="roc_auc",
    n_repeats=20, random_state=SEED, n_jobs=-1
)
importance = pd.DataFrame({
    "Feature": X_test.columns,
    "Mean importance": perm.importances_mean,
    "Standard deviation": perm.importances_std
}).sort_values("Mean importance", ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(importance["Feature"][::-1], importance["Mean importance"][::-1],
        xerr=importance["Standard deviation"][::-1], capsize=3)
ax.set_xlabel("Mean decrease in ROC-AUC after permutation")
ax.set_title("Top 15 features — permutation importance")
plt.tight_layout()
plt.show()
display(importance)

## 8. Unsupervised Exploration (Additional Analysis)

Clustering is exploratory and does not replace the supervised classifier. Features are standardized using the training set only. The number of clusters is set to 2 for this experiment, which is a strong assumption; diagnostic labels are used only after clustering to calculate external evaluation scores.

In [ ]:
X_train_scaled = StandardScaler().fit_transform(X_train)
kmeans = KMeans(n_clusters=2, n_init=20, random_state=SEED)
clusters = kmeans.fit_predict(X_train_scaled)
cluster_scores = pd.DataFrame([{
    "Silhouette": silhouette_score(X_train_scaled, clusters),
    "ARI vs. diagnosis": adjusted_rand_score(y_train, clusters),
    "NMI vs. diagnosis": normalized_mutual_info_score(y_train, clusters)
}], index=["K-means"])
display(cluster_scores.round(3))

In [ ]:
pca = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X_train_scaled)
pca_df = pd.DataFrame({"PC1": X_pca[:, 0], "PC2": X_pca[:, 1],
                       "Diagnosis": y_train.map({0: "Benign", 1: "Malignant"}).values,
                       "Cluster": clusters.astype(str)})
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="Diagnosis", alpha=.8, ax=axes[0])
axes[0].set_title("PCA projection — actual diagnosis")
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="Cluster", alpha=.8, ax=axes[1])
axes[1].set_title("PCA projection — K-means clusters")
for ax in axes:
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.tight_layout()
plt.show()

## 9. Conclusions and Limitations

### What this project demonstrates
- A reproducible workflow, from data quality checks to evaluation.
- Comparison of multiple models using stratified cross-validation.
- Hyperparameter selection restricted to the training set.
- Final evaluation using metrics suited to the malignant class, curves, and error analysis.
- An initial interpretability analysis and an additional unsupervised exploration.

### Limitations to state in any report
1. **Small and relatively old dataset:** 569 observations; performance may vary considerably on other data.
2. **No external validation:** no testing on an independent institution or population.
3. **Structured data only:** this project does not directly process MRI or mammography images.
4. **Decision threshold:** the default threshold of 0.5 may not be optimal; the choice depends on the context and must be validated.
5. **Bias and fairness:** the representativeness of subgroups is not studied here.
6. **Interpretability:** feature importance does not imply causality or provide clinical evidence.
7. **Use:** for learning purposes only, never for diagnosis or medical advice.

### Possible improvements for a research version
- External validation on an independent dataset, with the protocol specified in advance.
- Calibration analysis and bootstrap confidence intervals.
- Threshold analysis based on an objective defined with domain professionals.
- Comparison with regularized models and feature selection integrated into cross-validation.
- Reproducibility report covering library versions, protocol, random seeds, and limitations.

In [ ]:
# Reproducible summary of final results
summary_df = pd.DataFrame({
    "Metric": list(final_metrics.keys()),
    "Test value": list(final_metrics.values())
})
display(summary_df.style.format({"Test value": "{:.3f}"}).hide(axis="index"))
print("Final model:", search.best_estimator_.named_steps["model"])
print("Selected parameters:", search.best_params_)